# SPI / SPEI Computation — TerraClimate Global Data

Runs the full precipitation index pipeline in Google Colab:
1. Download per-year TerraClimate NetCDF files (cached on Google Drive to avoid repeat downloads)
2. Convert to a single Zarr store for fast chunked I/O
3. Compute SPI and/or SPEI using Dask + GPU (CuPy) with checkpoint resume
4. Generate probability layers and quality flags as Cloud-Optimised GeoTIFFs
5. Verify outputs and save everything to Google Drive

**Before running:** select `Runtime → Change runtime type → A100 GPU`.

**Disk budget (Colab Pro ~200 GB):**
| Region | NC input | Zarr store | SPI output |
|--------|----------|------------|------------|
| Full global (4320×8640) | ~5 GB/var | ~30 GB/var compressed | ~27 GB |
| Half res (2160×4320) | ~1.4 GB/var | ~8 GB/var | ~7 GB |
| Custom bbox | varies | varies | varies |

> TerraClimate NC files are automatically backed up to your Google Drive after the first download.
> On subsequent sessions they are restored from Drive instead of re-downloaded.


## Step 1 — Check GPU & Install Dependencies

Verifies the GPU runtime and installs all required Python packages.
CuPy is installed automatically matching the CUDA version on the current runtime.


In [3]:
import subprocess

# Verify GPU is available
try:
    result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                             '--format=csv,noheader'], capture_output=True, text=True)
    print('GPU:', result.stdout.strip())
except FileNotFoundError:
    print('WARNING: nvidia-smi not found. Make sure you selected a GPU runtime.')

# Detect CUDA version for matching CuPy wheel
try:
    r = subprocess.run(['nvcc', '--version'], capture_output=True, text=True)
    # Parse e.g. 'release 12.2' -> 122
    import re
    m = re.search(r'release (\d+)\.(\d+)', r.stdout)
    CUDA_MAJOR, CUDA_MINOR = (int(m.group(1)), int(m.group(2))) if m else (12, 2)
    print(f'CUDA: {CUDA_MAJOR}.{CUDA_MINOR}')
except Exception:
    CUDA_MAJOR, CUDA_MINOR = 12, 2
    print(f'Could not detect CUDA version, defaulting to {CUDA_MAJOR}.{CUDA_MINOR}')

# Map CUDA version to CuPy package name
cuda_tag = f'cuda{CUDA_MAJOR}{CUDA_MINOR}'
print(f'CuPy package: cupy-{cuda_tag}')

Could not detect CUDA version, defaulting to 12.2
CuPy package: cupy-cuda122


In [ ]:
import subprocess, sys

# Core scientific stack
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'xarray', 'zarr', 'dask[array]', 'netCDF4', 'scipy',
                'numpy', 'psutil', 'tqdm', 'rasterio'], check=True)

# CuPy — GPU-accelerated numpy (matches detected CUDA version)
try:
    cupy_pkg = f'cupy-{cuda_tag}'
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', cupy_pkg], check=True)
    import cupy as cp
    print(f'CuPy installed ({cupy_pkg}), GPU array test: OK')
except Exception as e:
    print(f'CuPy install failed ({e}) — will fall back to CPU')

print('All dependencies installed.')

CuPy install failed (Command '['/usr/bin/python3', '-m', 'pip', 'install', '-q', 'cupy-cuda128']' returned non-zero exit status 1.) — will fall back to CPU
All dependencies installed.


## Step 2 — Mount Google Drive

Mounts your Google Drive at `/content/drive`. Required for:
- Reading the `precip-index` source code from `src/`
- Caching downloaded NC files so they survive session resets
- Saving all outputs (NetCDF, GeoTIFF) for persistent storage


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted at /content/drive')

MessageError: Failed to issue request POST https://colab.research.google.com/tun/m/credentials-propagation/m-s-dftlim3puqgq?authtype=dfs_ephemeral&version=2&dryrun=false&propagate=true&record=false&authuser=0&authuser=0: Bad Request
Response body: 
<!DOCTYPE html>
<html lang=en>
  <meta charset=utf-8>
  <meta name=viewport content="initial-scale=1, minimum-scale=1, width=device-width">
  <title>Error 400 (Bad Request)!!1</title>
  <style>
    *{margin:0;padding:0}html,code{font:15px/22px arial,sans-serif}html{background:#fff;color:#222;padding:15px}body{margin:7% auto 0;max-width:390px;min-height:180px;padding:30px 0 15px}* > body{background:url(//www.google.com/images/errors/robot.png) 100% 5px no-repeat;padding-right:205px}p{margin:11px 0 22px;overflow:hidden}ins{color:#777;text-decoration:none}a img{border:0}@media screen and (max-width:772px){body{background:none;margin-top:0;max-width:none;padding-right:0}}#logo{background:url(//www.google.com/images/logos/errorpage/error_logo-150

## Step 3 — Configuration

**Edit this cell** to match your setup before running anything else.

Key settings:
- `DRIVE_REPO_PATH` — path to the `precip-index` repo folder on your Drive
- `YEAR_START / YEAR_END` — year range for TerraClimate download
- `CALIB_START / CALIB_END` — WMO calibration period (default 1991–2020)
- `RUN_SPI` / `SPEI_SCALES` — toggle which indices to compute
- `REGION` — `'full'` for global, `'half'` for half-resolution, `'bbox'` for a custom extent
- `SPI_CHUNK_SIZE` — spatial tile size; 180 is safe for an A100 (40 GB VRAM)


In [ ]:
# ============================================================
# CONFIGURATION — edit these values
# ============================================================

# Path to the precip-index repo on your Google Drive.
DRIVE_REPO_PATH = '/content/drive/MyDrive/01_projects/precip-index'

# Where to store downloaded NC files and outputs on Colab's local disk
WORK_DIR = '/content/precip_spi'

# Year range for TerraClimate download
YEAR_START = 1958
YEAR_END   = 2025

# WMO standard calibration period
CALIB_START = 1991
CALIB_END   = 2020

# SPI scale — set RUN_SPI = False to skip if already computed
SPI_SCALE = 12
RUN_SPI   = True

# SPEI scales to compute — set to [] to skip SPEI entirely
SPEI_SCALES = [12]

# Region to process.
# 'full'  → entire global grid (4320×8640)
# 'half'  → every-other pixel (2160×4320)
# 'bbox'  → custom bounding box defined below
REGION = 'full'

# Bounding box (used when REGION = 'bbox').
BBOX_LON_MIN =  90.0
BBOX_LON_MAX = 145.0
BBOX_LAT_MIN = -15.0
BBOX_LAT_MAX =  25.0

# Zarr chunk settings
ZARR_CHUNK_LAT = 180
ZARR_CHUNK_LON = 180

# SPI/SPEI processing chunk size — 180 is safe for A100 (40 GB VRAM)
SPI_CHUNK_SIZE = 180

# GPU flag
USE_GPU = True
# ============================================================

## Step 4 — Setup Working Directories & Source Code

Creates the local working tree under `WORK_DIR` and copies `src/` from your Drive
into the Colab session so Python can import `indices`, `chunked`, and `gpu`.
Re-run this cell if you update the source code on Drive mid-session.


In [ ]:
import os, sys, shutil

INPUT_DIR    = os.path.join(WORK_DIR, 'input')
OUTPUT_DIR   = os.path.join(WORK_DIR, 'output')
PPT_ZARR     = os.path.join(OUTPUT_DIR, 'ppt.zarr')
PET_ZARR     = os.path.join(OUTPUT_DIR, 'pet.zarr')
SPI_OUTPUT   = os.path.join(OUTPUT_DIR, f'spi_{SPI_SCALE}_month.nc')

for d in [INPUT_DIR, OUTPUT_DIR]:
    os.makedirs(d, exist_ok=True)

# Add the repo's src/ to sys.path so imports work
SRC_DIR = os.path.join(DRIVE_REPO_PATH, 'src')
if not os.path.isdir(SRC_DIR):
    raise FileNotFoundError(
        f"Repo src/ not found at {SRC_DIR}.\n"
        "Please upload the precip-index repo to your Drive and set DRIVE_REPO_PATH."
    )

LOCAL_SRC = os.path.join(WORK_DIR, 'src')
if os.path.exists(LOCAL_SRC):
    shutil.rmtree(LOCAL_SRC)
shutil.copytree(SRC_DIR, LOCAL_SRC)
sys.path.insert(0, LOCAL_SRC)

print(f'Working dir : {WORK_DIR}')
print(f'Input dir   : {INPUT_DIR}')
print(f'PPT Zarr    : {PPT_ZARR}')
print(f'PET Zarr    : {PET_ZARR}  {"(needed for SPEI)" if SPEI_SCALES else "(SPEI skipped)"}')
print(f'SPI output  : {SPI_OUTPUT}')
print(f'SPEI scales : {SPEI_SCALES if SPEI_SCALES else "skipped"}')

from gpu import GPU_AVAILABLE, gpu_info
print(f'\n{gpu_info()}')

Working dir : /content/precip_spi
Input dir   : /content/precip_spi/input
PPT Zarr    : /content/precip_spi/output/ppt.zarr
PET Zarr    : /content/precip_spi/output/pet.zarr  (needed for SPEI)
SPI output  : /content/precip_spi/output/spi_12_month.nc
SPEI scales : [1, 3, 6, 12]

GPU available: NVIDIA A100-SXM4-80GB (79.3 GB VRAM)


## Step 5 — Download TerraClimate Files

Downloads `TerraClimate_{var}_{year}.nc` files for `ppt` (and `pet` if SPEI is enabled).

**GDrive cache:** after the first download each file is copied to
`DRIVE_REPO_PATH/input/terraclimate/` on your Drive.  
On subsequent sessions files are restored from Drive instead of re-downloaded from the web,
saving ~5–10 GB of bandwidth and several minutes of wait time per variable.

Files already present on local disk are always skipped.


In [ ]:
import urllib.request, os, shutil
from tqdm.notebook import tqdm

BASE_URL  = 'https://climate.northwestknowledge.net/TERRACLIMATE-DATA'
MIN_SIZE  = 10 * 1024 * 1024   # 10 MB — valid TerraClimate files are larger
years     = range(YEAR_START, YEAR_END + 1)

variables = ['ppt']
if SPEI_SCALES:
    variables.append('pet')

# GDrive cache directory — NC files are stored here between sessions
DRIVE_NC_CACHE = os.path.join(DRIVE_REPO_PATH, 'input', 'terraclimate')
os.makedirs(DRIVE_NC_CACHE, exist_ok=True)

for var in variables:
    print(f'\n[{var}] Syncing {len(years)} files ({YEAR_START}–{YEAR_END}) ...')
    for year in tqdm(years, desc=var):
        fname = f'TerraClimate_{var}_{year}.nc'
        local = os.path.join(INPUT_DIR, fname)
        drive = os.path.join(DRIVE_NC_CACHE, fname)

        if os.path.exists(local) and os.path.getsize(local) >= MIN_SIZE:
            continue  # already on local disk

        if os.path.exists(drive) and os.path.getsize(drive) >= MIN_SIZE:
            # Copy from GDrive cache — much faster than re-downloading
            shutil.copy2(drive, local)
            continue

        # Download from NKN server
        try:
            urllib.request.urlretrieve(f'{BASE_URL}/{fname}', local)
        except Exception as e:
            print(f'  WARN: {fname} failed: {e}')
            if os.path.exists(local):
                os.remove(local)
            continue

        # Back up freshly-downloaded file to GDrive for future sessions
        shutil.copy2(local, drive)

    files     = sorted(f for f in os.listdir(INPUT_DIR) if f.startswith(f'TerraClimate_{var}_'))
    total_mb  = sum(os.path.getsize(os.path.join(INPUT_DIR, f)) for f in files) / 1e6
    cached    = sum(1 for y in years
                    if os.path.exists(os.path.join(DRIVE_NC_CACHE, f'TerraClimate_{var}_{y}.nc')))
    print(f'  {len(files)} files on local disk  |  {total_mb:.0f} MB  |  {cached}/{len(years)} cached on Drive')


## Step 6 — Convert NetCDF Files to Zarr

Merges all per-year files into a single Zarr store chunked as
`(time=all, lat=ZARR_CHUNK_LAT, lon=ZARR_CHUNK_LON)`.  
Compression uses Blosc/zstd with bitshuffle — typically 3–4× ratio on climate float32 data.

NC files are deleted from local disk after a successful write to reclaim space.
The Zarr store is kept for SPI/SPEI processing in Step 7.

Skip this cell if the Zarr store already exists (message will confirm).


In [ ]:
import glob, time, gc, shutil
import numpy as np
import xarray as xr
import zarr

def disk_usage_gb(path='/content'):
    """Return (used_gb, free_gb) for the filesystem containing path."""
    import shutil as _sh
    s = _sh.disk_usage(path)
    return s.used / 1e9, s.free / 1e9

def print_disk():
    used, free = disk_usage_gb()
    print(f'  Disk: {used:.1f} GB used  |  {free:.1f} GB free  |  ~200 GB cap')

# numcodecs.Blosc compressor — works with zarr format 2, which is what
# xarray's zarr backend expects. zarr v3 defaults to format 3 but xarray's
# open_zarr reliably reads format 2 stores on all zarr versions.
from numcodecs import Blosc
ZARR_COMPRESSOR = Blosc(cname='zstd', clevel=3, shuffle=Blosc.BITSHUFFLE)

def convert_to_zarr(var, zarr_store, delete_nc_after=True):
    files = sorted(glob.glob(os.path.join(INPUT_DIR, f'TerraClimate_{var}_*.nc')))
    if not files:
        raise RuntimeError(f'No {var} files found in {INPUT_DIR}')
    print(f'\n[{var}] Found {len(files)} files.')
    print_disk()

    if os.path.exists(zarr_store):
        print(f'  Zarr store already exists: {zarr_store}  (delete to rebuild)')
        return

    print(f'  Opening with open_mfdataset ...', end=' ', flush=True)
    t0 = time.perf_counter()
    ds = xr.open_mfdataset(
        files, combine='by_coords',
        chunks={'time': 12, 'lat': ZARR_CHUNK_LAT, 'lon': ZARR_CHUNK_LON}
    )
    print(f'{time.perf_counter()-t0:.1f}s')

    if REGION == 'bbox':
        ds = ds.sel(lat=slice(BBOX_LAT_MAX, BBOX_LAT_MIN),
                    lon=slice(BBOX_LON_MIN, BBOX_LON_MAX))
    elif REGION == 'half':
        ds = ds.isel(lat=slice(None, None, 2), lon=slice(None, None, 2))

    n_time, n_lat, n_lon = ds.sizes['time'], ds.sizes['lat'], ds.sizes['lon']
    est_raw_gb = n_time * n_lat * n_lon * 4 / 1e9
    print(f'  Shape: time={n_time}, lat={n_lat}, lon={n_lon}  '
          f'(raw float32: ~{est_raw_gb:.1f} GB  →  ~{est_raw_gb/4:.1f} GB compressed est.)')

    tile = ds[[var]].chunk({'time': n_time, 'lat': ZARR_CHUNK_LAT,
                            'lon': ZARR_CHUNK_LON}).astype(np.float32)
    for v in tile.data_vars:
        tile[v].encoding.clear()

    encoding = {v: {'compressor': ZARR_COMPRESSOR} for v in tile.data_vars}

    print(f'  Writing → {zarr_store} (compressed) ...', flush=True)
    t0 = time.perf_counter()
    tile.to_zarr(zarr_store, mode='w', encoding=encoding, zarr_format=2)
    elapsed = time.perf_counter() - t0
    zarr_gb = sum(os.path.getsize(os.path.join(r, f))
                  for r, _, fs in os.walk(zarr_store) for f in fs) / 1e9
    print(f'  Done  |  {elapsed:.1f}s  |  {zarr_gb:.2f} GB on disk  '
          f'(ratio: {est_raw_gb/zarr_gb:.1f}×)')

    del ds, tile
    gc.collect()

    # Delete NC files to reclaim disk space
    if delete_nc_after:
        print(f'  Deleting {len(files)} NC files to free space ...', end=' ', flush=True)
        freed = sum(os.path.getsize(f) for f in files) / 1e9
        for f in files:
            os.remove(f)
        print(f'{freed:.1f} GB freed')

    print_disk()


convert_to_zarr('ppt', PPT_ZARR, delete_nc_after=True)

if SPEI_SCALES:
    convert_to_zarr('pet', PET_ZARR, delete_nc_after=True)
else:
    print('\nSPEI skipped — PET Zarr not needed.')

## Step 7 — Compute SPI & SPEI

Reads from the Zarr store and fits a gamma distribution pixel-by-pixel using
spatial tiles of `SPI_CHUNK_SIZE × SPI_CHUNK_SIZE` degrees.  
GPU acceleration (CuPy) is used automatically when available.

**Checkpoint resume:** a `.ckpt` file is written after each tile.
If the session crashes, re-run this cell — completed tiles are skipped.

Outputs saved to `OUTPUT_DIR` and immediately copied to `DRIVE_REPO_PATH/output/colab/`:
- `spi_{scale}_month.nc` + `spi_{scale}_month_params.nc`
- `spei_{scale}_month.nc` + `spei_{scale}_month_params.nc` (if SPEI enabled)


In [ ]:
import time, shutil, os, xarray as xr, pathlib, json
from gpu import gpu_info
from indices import spi_global, spei_global

DISK_CAP_GB  = 200.0
DISK_WARN_GB = 20.0   # warn when less than this is free

print(f'Device : {gpu_info()}')
print(f'Chunks : {SPI_CHUNK_SIZE}×{SPI_CHUNK_SIZE}  GPU: {USE_GPU}')
print_disk()
print()

DRIVE_OUTPUT_DIR = os.path.join(DRIVE_REPO_PATH, 'output', 'colab')
os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)

def check_disk(label):
    used, free = disk_usage_gb()
    print(f'  [{label}] Disk: {used:.1f} GB used  |  {free:.1f} GB free')
    if free < DISK_WARN_GB:
        raise RuntimeError(
            f'Less than {DISK_WARN_GB:.0f} GB free ({free:.1f} GB) — '
            f'aborting before starting {label} to avoid filling disk.'
        )

def run_index(label, fn, output_path, **kwargs):
    check_disk(label)
    ckpt = pathlib.Path(output_path + '.ckpt')
    if ckpt.exists():
        done = json.loads(ckpt.read_text()).get('done', [])
        print(f'  Resuming from checkpoint: {len(done)} chunks already done.')
    def on_chunk(chunk_info, pct):
        print(f'  [{pct:5.1f}%] {chunk_info}', flush=True)
    print(f'=== {label} → {os.path.basename(output_path)} ===')
    t0 = time.perf_counter()
    fn(output_path=output_path, callback=on_chunk, **kwargs)
    elapsed = time.perf_counter() - t0
    print(f'{label} done in {elapsed:.1f}s  ({elapsed/60:.1f} min)')
    for src in [output_path, output_path.replace('.nc', '_params.nc')]:
        if os.path.exists(src):
            dest = os.path.join(DRIVE_OUTPUT_DIR, os.path.basename(src))
            shutil.copy2(src, dest)
            print(f'  Saved → Drive: {os.path.basename(dest)}  '
                  f'({os.path.getsize(dest)/1e6:.0f} MB)')
    print_disk()
    print()

# ---- SPI ----
if RUN_SPI:
    ds_ppt = xr.open_zarr(PPT_ZARR)
    run_index(
        f'SPI-{SPI_SCALE}', spi_global, SPI_OUTPUT,
        precip_path=ds_ppt,
        scale=SPI_SCALE,
        calibration_start_year=CALIB_START,
        calibration_end_year=CALIB_END,
        chunk_size=SPI_CHUNK_SIZE,
        distribution='gamma',
        save_params=True,
        use_gpu=USE_GPU,
    )
else:
    print(f'SPI-{SPI_SCALE} skipped (RUN_SPI=False).')
    print(f'  Exists: {os.path.exists(SPI_OUTPUT)}\n')

# ---- SPEI ----
if SPEI_SCALES:
    ds_ppt = xr.open_zarr(PPT_ZARR)
    ds_pet = xr.open_zarr(PET_ZARR)
    for scale in SPEI_SCALES:
        spei_out = os.path.join(OUTPUT_DIR, f'spei_{scale}_month.nc')
        run_index(
            f'SPEI-{scale}', spei_global, spei_out,
            precip_path=ds_ppt,
            pet_path=ds_pet,
            scale=scale,
            calibration_start_year=CALIB_START,
            calibration_end_year=CALIB_END,
            chunk_size=SPI_CHUNK_SIZE,
            distribution='gamma',
            save_params=True,
            use_gpu=USE_GPU,
        )
else:
    print('SPEI skipped (SPEI_SCALES is empty).')

## Step 8 — Probability Layers + Quality Flag → GeoTIFF

Computes three drought-probability bands and a data-quality flag from the SPI/SPEI output,
then writes a 4-band Cloud-Optimised GeoTIFF (EPSG:4326, Deflate compressed).

**Bands:**
| Band | Name | Description |
|------|------|-------------|
| 1 | `prob_{index}_neg1p0` | P(index ≤ −1.0) — moderate drought or worse |
| 2 | `prob_{index}_neg1p5` | P(index ≤ −1.5) — severe drought or worse |
| 3 | `prob_{index}_neg2p0` | P(index ≤ −2.0) — extreme drought |
| 4 | `quality_flag` | 1 = high / 2 = medium / 3 = low (based on precipitation record) |

Output is saved to `OUTPUT_DIR` and copied to `DRIVE_REPO_PATH/output/colab/`.
Intermediate band arrays are cached under `OUTPUT_DIR/.prob_cache/` so re-runs are fast.


In [22]:
import gc, sys, shutil
from pathlib import Path
import numpy as np
import xarray as xr
import rasterio
from rasterio.crs import CRS
from rasterio.transform import from_bounds

THRESHOLDS = [-1.0, -1.5, -2.0]
NODATA     = -9999.0
CACHE_DIR  = Path(OUTPUT_DIR) / '.prob_cache'
CACHE_DIR.mkdir(parents=True, exist_ok=True)

DRIVE_OUTPUT_DIR = os.path.join(DRIVE_REPO_PATH, 'output', 'colab')
os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)

QUALITY_HIGH_SAMPLES   = 30
QUALITY_MEDIUM_SAMPLES = 10
QUALITY_HIGH_PRECIP    = 200
QUALITY_MEDIUM_PRECIP  = 100


def compute_prob_geotiff(index_name, scale, nc_path):
    label        = f'{index_name.upper()}-{scale}'
    cache_probs  = CACHE_DIR / f'{index_name}_{scale}_prob_bands.npy'
    cache_quality= CACHE_DIR / 'quality.npy'
    cache_coords = CACHE_DIR / 'coords.npz'

    if not os.path.exists(nc_path):
        print(f'[{label}] Output file not found: {nc_path} — skipping.')
        return

    print(f'\n{"="*55}')
    print(f'PROB LAYERS: {label}')
    print(f'{"="*55}')

    # ---- Step 1: Probability maps ----
    index_var = f'{index_name}_gamma_{scale}_month'

    if cache_probs.exists() and cache_coords.exists():
        print('  Loading cached probability bands...')
        prob_bands = list(np.load(cache_probs))
        coords = np.load(cache_coords)
        lat, lon = coords['lat'], coords['lon']
        nlat, nlon = len(lat), len(lon)
    else:
        ds = xr.open_dataset(nc_path, chunks={'time': -1, 'lat': 60, 'lon': 1440})
        da = ds[index_var]
        nlat, nlon, n_time = da.sizes['lat'], da.sizes['lon'], da.sizes['time']
        lat, lon = ds['lat'].values, ds['lon'].values
        print(f'  Shape: {da.shape}')

        # Count valid (non-NaN) time steps per pixel — used as denominator
        valid_count = da.notnull().sum('time').compute().values.astype(np.float32)
        has_data    = valid_count > 0   # mask: True where pixel has any valid data

        prob_bands = []
        for thresh in THRESHOLDS:
            name = f"prob_{str(thresh).replace('-','neg').replace('.','p')}"
            print(f'  Computing {name} ...', end=' ', flush=True)
            below = (da <= thresh).sum('time').compute().values.astype(np.float32)
            # Divide by valid count; set nodata pixels to NaN
            prob = np.where(has_data, below / np.where(has_data, valid_count, 1), np.nan)
            prob_bands.append(prob.astype(np.float32))
            valid_vals = prob[has_data]
            print(f'range {valid_vals.min():.4f}–{valid_vals.max():.4f}  '
                  f'({(~has_data).sum():,} nodata pixels)')

        ds.close(); gc.collect()
        np.save(cache_probs, np.stack(prob_bands))
        np.savez(cache_coords, lat=lat, lon=lon)
        print('  Cached.')

    # ---- Step 2: Quality flags ----
    if cache_quality.exists():
        print('  Loading cached quality flag...')
        quality = np.load(cache_quality).astype(np.float32)
        # Re-apply nodata mask to quality flag using first prob band
        nodata_mask = np.isnan(prob_bands[0])
        quality[nodata_mask] = NODATA
    else:
        print('  Computing quality flags from Zarr store (NC files were deleted)...')
        years = range(YEAR_START, YEAR_END + 1)
        nonzero_count = np.zeros((nlat, nlon), dtype=np.int32)
        total_precip  = np.zeros((nlat, nlon), dtype=np.float64)
        _ds_ppt_zarr = xr.open_zarr(PPT_ZARR)
        da_ppt = _ds_ppt_zarr['ppt']
        for year in years:
            sys.stdout.write(f'\r  {year} ({year - YEAR_START + 1}/{len(years)})...')
            sys.stdout.flush()
            mask = (da_ppt.time.dt.year == year).values
            ppt = da_ppt.isel(time=mask).values
            nonzero_count += (ppt > 0).sum(axis=0).astype(np.int32)
            total_precip  += np.nansum(ppt, axis=0)
            del ppt; gc.collect()
        _ds_ppt_zarr.close()
        print()
        mean_annual = total_precip / len(years)
        quality_int = np.full((nlat, nlon), 3, dtype=np.uint8)
        medium_mask = (nonzero_count >= QUALITY_MEDIUM_SAMPLES) & (mean_annual >= QUALITY_MEDIUM_PRECIP)
        quality_int[medium_mask] = 2
        high_mask = (nonzero_count >= QUALITY_HIGH_SAMPLES) & (mean_annual >= QUALITY_HIGH_PRECIP)
        quality_int[high_mask] = 1
        print(f'  High={high_mask.sum():,}  Medium={(medium_mask.sum()-high_mask.sum()):,}  Low={(~medium_mask).sum():,}')
        del nonzero_count, total_precip, mean_annual, medium_mask, high_mask; gc.collect()
        np.save(cache_quality, quality_int)

        # Apply nodata mask
        quality = quality_int.astype(np.float32)
        nodata_mask = np.isnan(prob_bands[0])
        quality[nodata_mask] = NODATA
        print('  Cached.')

    # ---- Step 3: Write GeoTIFF ----
    geotiff_out = os.path.join(OUTPUT_DIR, f'{index_name}{scale}_prob_quality.tif')
    res_lat = abs(lat[1] - lat[0])
    res_lon = abs(lon[1] - lon[0])
    transform = from_bounds(
        lon.min() - res_lon/2, lat.min() - res_lat/2,
        lon.max() + res_lon/2, lat.max() + res_lat/2,
        nlon, nlat,
    )
    band_meta = [
        (f'prob_{index_name}_neg1p0', f'P({label} ≤ -1.0) moderate drought or worse'),
        (f'prob_{index_name}_neg1p5', f'P({label} ≤ -1.5) severe drought or worse'),
        (f'prob_{index_name}_neg2p0', f'P({label} ≤ -2.0) extreme drought'),
        ('quality_flag',             '1=high 2=medium 3=low data quality'),
    ]
    # Replace NaN with NODATA for GeoTIFF
    all_bands = [np.where(np.isnan(b), NODATA, b) for b in prob_bands] + [quality]

    print(f'  Writing GeoTIFF → {os.path.basename(geotiff_out)} ...', flush=True)
    with rasterio.open(
        geotiff_out, 'w', driver='GTiff',
        height=nlat, width=nlon, count=4, dtype='float32',
        crs=CRS.from_epsg(4326), transform=transform,
        compress='deflate', predictor=2, tiled=True,
        blockxsize=256, blockysize=256, BIGTIFF='YES',
        nodata=NODATA,
    ) as dst:
        for i, (data, (bname, bdesc)) in enumerate(zip(all_bands, band_meta), start=1):
            dst.write(data, i)
            dst.update_tags(i, name=bname, description=bdesc)
        dst.update_tags(
            source=f'precip-index / TerraClimate {label} {YEAR_START}-{YEAR_END}',
            calibration=f'{CALIB_START}-{CALIB_END}',
            thresholds=str(THRESHOLDS),
            nodata=str(NODATA),
        )

    size_mb = Path(geotiff_out).stat().st_size / 1e6
    print(f'  {size_mb:.0f} MB  ✓')

    dest = os.path.join(DRIVE_OUTPUT_DIR, os.path.basename(geotiff_out))
    shutil.copy2(geotiff_out, dest)
    print(f'  Saved → Drive: {os.path.basename(dest)}')


# ---- Run for SPI ----
compute_prob_geotiff('spi', SPI_SCALE, SPI_OUTPUT)

# ---- Run for each SPEI scale ----
for scale in SPEI_SCALES:
    spei_out = os.path.join(OUTPUT_DIR, f'spei_{scale}_month.nc')
    compute_prob_geotiff('spei', scale, spei_out)


PROB LAYERS: SPI-12
  Shape: (816, 4320, 8640)


/tmp/ipykernel_780/2236595171.py:47: UserWarning: The specified chunks separate the stored chunks along dimension "lat" starting at index 60. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_dataset(nc_path, chunks={'time': -1, 'lat': 60, 'lon': 1440})


  Computing prob_neg1p0 ... range 0.0000–0.5863  (24,792,188 nodata pixels)
  Computing prob_neg1p5 ... range 0.0000–0.4708  (24,792,188 nodata pixels)
  Computing prob_neg2p0 ... range 0.0000–0.3640  (24,792,188 nodata pixels)
  Cached.
  Computing quality flags from PPT files...
  1958 (1/68)...

FileNotFoundError: [Errno 2] No such file or directory: '/content/precip_spi/input/TerraClimate_ppt_1958.nc'

## Step 9 — Verify Outputs

Prints a quick summary of every output file produced in this session:
shape, time range, valid pixel count, value range, and file size.
All outputs should already have been copied to Google Drive by Steps 7 and 8.
Run this cell after all processing is complete to confirm everything looks correct.


In [ ]:
import xarray as xr, numpy as np, os
from pathlib import Path

def verify_nc(path, label):
    if not os.path.exists(path):
        print(f'  MISSING: {label} → {path}')
        return
    ds = xr.open_dataset(path)
    var = list(ds.data_vars)[0]
    da  = ds[var]
    last = da.isel(time=-1).values
    valid = last[~np.isnan(last)]
    size_mb = os.path.getsize(path) / 1e6
    print(f'  {label}')
    print(f'    shape  : {dict(zip(da.dims, da.shape))}')
    print(f'    time   : {str(da.time.values[0])[:10]} → {str(da.time.values[-1])[:10]}')
    print(f'    last   : {len(valid):,} valid pixels  '
          f'min={valid.min():.2f}  mean={valid.mean():.2f}  max={valid.max():.2f}')
    print(f'    size   : {size_mb:.0f} MB')
    ds.close()

def verify_tif(path, label):
    if not os.path.exists(path):
        print(f'  MISSING: {label} → {path}')
        return
    import rasterio
    with rasterio.open(path) as src:
        print(f'  {label}')
        print(f'    bands  : {src.count}  size: {src.width}×{src.height}  CRS: {src.crs.to_epsg()}')
        print(f'    size   : {os.path.getsize(path)/1e6:.0f} MB')

print('=== Output verification ===')

# SPI
if RUN_SPI:
    verify_nc(SPI_OUTPUT, f'SPI-{SPI_SCALE}')
    verify_nc(SPI_OUTPUT.replace('.nc', '_params.nc'), f'SPI-{SPI_SCALE} params')

# SPEI
for scale in SPEI_SCALES:
    spei_out = os.path.join(OUTPUT_DIR, f'spei_{scale}_month.nc')
    verify_nc(spei_out, f'SPEI-{scale}')
    verify_nc(spei_out.replace('.nc', '_params.nc'), f'SPEI-{scale} params')

# GeoTIFFs
if RUN_SPI:
    verify_tif(os.path.join(OUTPUT_DIR, f'spi{SPI_SCALE}_prob_quality.tif'), f'SPI-{SPI_SCALE} GeoTIFF')
for scale in SPEI_SCALES:
    verify_tif(os.path.join(OUTPUT_DIR, f'spei{scale}_prob_quality.tif'), f'SPEI-{scale} GeoTIFF')

print()
print('=== Drive output directory ===')
drive_out = os.path.join(DRIVE_REPO_PATH, 'output', 'colab')
if os.path.isdir(drive_out):
    for f in sorted(os.listdir(drive_out)):
        size_mb = os.path.getsize(os.path.join(drive_out, f)) / 1e6
        print(f'  {f:<50}  {size_mb:>8.0f} MB')
else:
    print(f'  Drive output dir not found: {drive_out}')
